# 実行のトレースを調べる

処理のどこに時間がかかったかを調べるためのHow-toです。[SDKのインストール](../getting_started/install.md)だけで最初の例を実行できます。Notebookでは同じ環境のカーネルを選んでください。ExperimentやAdapterの導入は前提ではありません。

以下の表示時間はこの一回の実行の観測値であり、性能比較や改善の根拠ではありません。

まず`%%ommx_trace`でセルの経過時間を表示します。出力の**スパン**は、名前と開始・終了時刻を持つ一つの処理区間です。内部の処理にもスパンが記録されていれば、親子になった区間から時間の内訳を読めます。任意のPython文が自動で個別に計測されるわけではありません。OMMXはこの情報を[OpenTelemetry](https://opentelemetry.io/)で出力し、{mod}`ommx.tracing`で表示・保存できます。

- **`%%ommx_trace`** — 単一セルの実行中に発生したスパンをネストしたテキストツリーとして表示するJupyterセルマジック。加えてChrome Trace Event Format形式のJSONダウンロードリンクも表示されます。
- **{class}`~ommx.tracing.capture_trace` / {func}`~ommx.tracing.traced`** — 同じ仕組みを通常のPythonスクリプト、テスト、CIから利用するためのコンテキストマネージャとデコレータ。

ローカルでの表示に追加のOTel exporterは不要です。外部へ送信したい場合だけ、後述の[外部OTelバックエンドへ送信する](#own-tracer-provider)を参照してください。

## クイックツアー

### セルマジック (`%%ommx_trace`)

ノートブックごとに一度、拡張を読み込みます（通常は最初のセル）。

In [1]:
%load_ext ommx.tracing

以降、任意のセルの先頭に `%%ommx_trace` を付けるだけで計測されます。

In [2]:
%%ommx_trace
from ommx import DecisionVariable, Instance, Sense

x = DecisionVariable.binary(0, name="x")
y = DecisionVariable.binary(1, name="y")
instance = Instance.from_components(
    decision_variables=[x, y],
    objective=x + y,
    constraints={},
    sense=Sense.Maximize,
)
solution = instance.evaluate({0: 1.0, 1: 1.0})

セルの出力として以下の2つが表示されます。

1. セル内で記録されたスパンを**ネストしたテキストツリー**として描画。各ノードには経過時間と代表的な属性が付与されます。上の出力では、セル全体を囲む`ommx_trace_cell`だけが表示されています。これはセル全体の時間であり、`evaluate`だけの時間ではありません。
2. Chrome Trace Event Format形式のトレース全体の**ダウンロードリンク**。生成されたJSONファイルを [Perfetto](https://ui.perfetto.dev/) / [speedscope](https://www.speedscope.app/) / `chrome://tracing` に読み込ませることでフレームグラフとして閲覧できます。

セル内で例外が発生した場合も、トレースは表示され、失敗した区間には`[ERROR]`が付きます。元の例外もそのまま伝わるので、トレースが表示されても処理の成功を意味しません。

### コンテキストマネージャ (`capture_trace`)

通常のPythonスクリプトからも同じ仕組みを利用できます。

In [3]:
from ommx.tracing import capture_trace, save_chrome_trace
from ommx import DecisionVariable, Instance, Sense

x = DecisionVariable.binary(0, name="x")
y = DecisionVariable.binary(1, name="y")

instance = Instance.from_components(
    decision_variables=[x, y],
    objective=x + y,
    constraints={},
    sense=Sense.Maximize,
)

with capture_trace() as trace:
    solution = instance.evaluate({0: 1.0, 1: 1.0})

trace

└── ommx_trace_block (29.0 µs) {scope=ommx.tracing}

`trace`は計測した区間の記録です。Notebookでは最後に`trace`を置くと表示でき、通常のPythonスクリプトでは`print(trace)`でテキストツリーを表示できます。上の出力は`with`ブロック全体の経過時間を表しています。

次のセルでChrome Trace JSONとして保存します。Notebookの外で内訳を調べたり、ほかの人に渡したりできます。

記録の構造や表示・保存のAPIは、{class}`~ommx.tracing.TraceResult`、{func}`~ommx.tracing.render_text_tree`、{func}`~ommx.tracing.save_chrome_trace`を参照してください。

ブロック内で例外が発生しても、外側の`except`や`finally`から`trace`を調査・保存できます。元の例外はそのまま伝わります。

In [4]:
import tempfile
from pathlib import Path

output_path = Path(tempfile.gettempdir()) / "ommx_trace.json"
save_chrome_trace(trace, output_path)

### デコレータ (`@traced`)

{func}`~ommx.tracing.traced` は {class}`~ommx.tracing.capture_trace` の糖衣構文です。

In [5]:
import tempfile
from pathlib import Path

from ommx.tracing import traced

evaluate_output = Path(tempfile.gettempdir()) / "evaluate_trace.json"

@traced(output=str(evaluate_output))
def evaluate_once(inst):
    return inst.evaluate({0: 1.0, 1: 1.0})

solution = evaluate_once(instance)

同じ関数を繰り返し計測する場合は、上のようにデコレータを付けると呼び出すたびに記録を保存できます。呼び出し形式や名前の指定は{func}`~ommx.tracing.traced`を参照してください。

例外が発生した場合も保存を試みますが、保存に失敗しても元の例外を上書きしません。そのため、例外時の保存はbest-effortであり、必ずファイルが残る保証ではありません。

(adapter-trace-breakdown)=
## 任意：Adapterの処理の内訳を読む

求解のどの段階に時間がかかったかを調べる例です。この節だけ{ref}`HiGHS Adapterを追加 <install-highs-adapter>`してください。Adapterは`Instance`をソルバーに渡して求解するための窓口で、以下の`solve`は得られた候補を`Solution`として返します。Experimentは使いません。

In [6]:
from ommx_highs_adapter import OMMXHighsAdapter

with capture_trace("highs_solve") as solver_trace:
    solution = OMMXHighsAdapter.solve(instance, verbose=False)

solver_trace

└── highs_solve (7.44 ms) {scope=ommx.tracing}
    └── solve (6.76 ms) {scope=ommx.adapter.highs} [adapter='ommx_highs_adapter.adapter.OMMXHighsAdapter']
        ├── convert (3.68 ms) {scope=ommx.adapter.highs}
        ├── call (2.79 ms) {scope=ommx.adapter.highs}
        └── decode (191.0 µs) {scope=ommx.adapter.highs}

`solve`の下に、問題をソルバー用に組み立てる`convert`、ソルバーを呼ぶ`call`、結果をOMMXの候補に戻す`decode`が並びます。親の時間には子の時間も含まれるため、全行を足した値が全体の所要時間になるわけではありません。`{scope=...}`はスパンを記録したpackageやmodule、`adapter`属性は使ったAdapterを示します。

このように、内訳を調べられるのはスパンを記録する処理です。最初の評価例のように外側の区間だけが表示される場合、その出力から内部の内訳を判断することはできません。

(experiment-run-trace-storage)=
## 任意：ExperimentにRunトレースを保存する

実験記録と一緒にトレースを残したい場合は、[Experiment Tutorial](../tutorial/experiment_management)で`Experiment`と`Run`の記録方法を確認してください。この節では最初の`instance`の候補を評価して記録します。上のHiGHSの節は実行しなくてかまいません。

`store_trace=True`を付けてExperimentを作ると、各Runの区間も保存できます。

In [7]:
from ommx.experiment import Experiment

with Experiment.with_temp_local_registry(store_trace=True) as experiment:
    with experiment.run() as run:
        solution = instance.evaluate({0: 1.0, 1: 1.0})
        run.log_solution("candidate", solution)
        run.log_parameter("objective", solution.objective)

trace = experiment.runs[0].trace
assert trace is not None
trace

└── Run (6.16 ms) {scope=ommx.experiment} [ommx.run.id=0]

`Run`は`with experiment.run() as run:`の区間を表します。`experiment.run()`を呼び出す瞬間だけの時間ではありません。`ommx.run.id`属性はどのRunかを示します。保存したトレースは、Experimentを読み直した後も{attr}`~ommx.experiment.SealedRun.trace`から表示できます。

In [8]:
loaded = Experiment.from_artifact(experiment.artifact)
trace = loaded.runs[0].trace
assert trace is not None
trace

└── Run (6.16 ms) {scope=ommx.experiment} [ommx.run.id=0]

保存されるのはRunの区間です。Experiment全体の寿命やNotebookのセル間の待ち時間を記録したものではありません。設定の詳細は{class}`~ommx.experiment.Experiment`、継続する実験の扱いは{meth}`~ommx.experiment.Experiment.fork`を参照してください。

(own-tracer-provider)=
## 外部OTelバックエンドへ送信する

Run trace storageは、外部OTelバックエンドへのexport設定とは独立しています。この任意の例だけは追加パッケージ`opentelemetry-exporter-otlp-proto-grpc`が必要です。先に、SDKを導入した環境で`python -m pip install opentelemetry-exporter-otlp-proto-grpc`を実行してください。送信先のOTLP endpointと必要な認証は、利用する外部サービスに合わせて設定します。ここまでのローカル表示には不要です。

OTLP、Jaeger、Honeycombなどへスパンを送信したい場合は、OMMXへの最初の呼び出し前にSDK TracerProviderを設定し、そこへexporterを追加してください。

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)

from ommx import Instance
```

`%%ommx_trace`、{class}`~ommx.tracing.capture_trace`、Experimentのtrace storageは、アクティブなSDK providerにcollectorを追加します。このため同じスパンをNotebook上で表示しつつ、外部バックエンドにも送信できます。providerの設定は、スクリプトやNotebook kernelの先頭で、OMMXをimportまたは呼び出す前に行ってください。